In [5]:
from google_play_scraper import reviews, Sort
import pandas as pd

In [6]:
CBE_APP_ID = "com.combanketh.mobilebanking"
BOA_APP_ID = "com.boa.boaMobileBanking"
DASHEN_APP_ID = "com.dashen.dashensuperapp"

In [ ]:
def scrape_bank_reviews(app_id, bank_name, count=500):
    """
    Scrape reviews from Google Play Store.
    """

    result, continuation_token = reviews(
        app_id,
        lang="en",
        country="et",
        sort=Sort.NEWEST,
        count=count
    )

    review_data = []

    for review in result:
        review_data.append({
            "review_id": review["reviewId"],
            "review": review["content"],
            "rating": review["score"],
            "date": review["at"].strftime("%Y-%m-%d"),
            "bank": bank_name,
            "source": "Google Play"
        })

    df = pd.DataFrame(review_data)

    return df

In [12]:
# Commercial Bank of Ethiopia
df_cbe = scrape_bank_reviews(
    CBE_APP_ID,
    "Commercial Bank of Ethiopia",
    count=500
)

# Bank of Abyssinia
df_boa = scrape_bank_reviews(
    BOA_APP_ID,
    "Bank of Abyssinia",
    count=500
)

# Dashen Bank
df_dashen = scrape_bank_reviews(
    DASHEN_APP_ID,
    "Dashen Bank",
    count=500
)

In [14]:
df_reviews = pd.concat(
    [df_cbe, df_boa, df_dashen],
    ignore_index=True
)

print("Combined dataset shape:", df_reviews.shape)

df_reviews.head()

Combined dataset shape: (1500, 6)


,review_id,review,rating,date,bank,source
0,363a5616-ed3d-4274-85ee-77071067f81d,wow,5,2026-05-13,Commercial Bank of Ethiopia,Google Play
1,56185597-d29b-4a60-a0fb-6783638230a7,Good application,2,2026-05-13,Commercial Bank of Ethiopia,Google Play
2,35efe702-40c9-4e46-ad67-7574ab9ef42d,"Nice, but I can't get some recently transactio...",5,2026-05-13,Commercial Bank of Ethiopia,Google Play
3,b41cb49d-59d3-41b8-bbb2-b1952005951b,Very Secure but very poor interface and limite...,1,2026-05-13,Commercial Bank of Ethiopia,Google Play
4,22026bb2-c9c4-4040-892b-2a77ebee48b7,very nice 100%,5,2026-05-13,Commercial Bank of Ethiopia,Google Play


In [10]:
# Initial dataset shape
print("Initial dataset shape:", df_reviews.shape)

# Check missing values
print("\nMissing values before cleaning:")
print(df_reviews.isnull().sum())

# Store initial rows
initial_rows = len(df_reviews)

# Remove rows with missing review or rating
df_reviews = df_reviews.dropna(
    subset=["review", "rating"]
)

missing_removed = initial_rows - len(df_reviews)

print(f"\nRows removed due to missing values: {missing_removed}")

# Remove duplicate reviews using review_id
before_duplicates = len(df_reviews)

df_reviews = df_reviews.drop_duplicates(
    subset=["review_id"]
)

duplicates_removed = before_duplicates - len(df_reviews)

print(f"Duplicate rows removed: {duplicates_removed}")

Initial dataset shape: (1500, 6)

Missing values before cleaning:
review_id    0
review       0
rating       0
date         0
bank         0
source       0
dtype: int64

Rows removed due to missing values: 0
Duplicate rows removed: 0


In [15]:
# Reviews per bank
print(df_reviews["bank"].value_counts())

# Total reviews
print("\nTotal reviews collected:", len(df_reviews))

bank
Commercial Bank of Ethiopia    500
Bank of Abyssinia              500
Dashen Bank                    500
Name: count, dtype: int64

Total reviews collected: 1500


In [17]:
final_df = df_reviews[
    [
        "review_id",
        "review",
        "rating",
        "date",
        "bank",
        "source"
    ]
]

final_df.to_csv(
    "../data/raw/bank_reviews_cleaned.csv",
    index=False
)

print("Final cleaned dataset saved successfully.")

Final cleaned dataset saved successfully.
